# Satellite Overlay Visualization — Northeast India Holdout
**Runtime:** Google Colab — **GPU (T4 or A100)**

**Purpose:** Publication-quality figures with **satellite imagery as the base layer** and
**semi-transparent coloured patch rectangles** overlaid showing predicted/observed connectivity.

The workflow:
1. Load `sampled_patches.csv` + NB17 model predictions
2. Use `contextily` to pull Esri World Imagery satellite basemap tiles on-the-fly
3. Plot satellite base → overlay 6.72 km chip rectangles coloured by value

**Main figures:**
- **(A) Observed** ground truth (`log_mean_tests`) on satellite
- **(B) Predicted** (Prithvi-only / Fusion) on satellite
- **(C) Residuals** (predicted − true) on satellite

**Optional:** Binary overlay (observed vs predicted `has_coverage`).

**Prerequisites:** Run NB17 first for saved XGBoost models.

**Inputs:**
- `data/raw/patches_2019/` — GeoTIFF patches (for Prithvi embedding extraction)
- `data/processed/sampled_patches.csv` — metadata + labels
- `data/raw/ookla_india_2019_Q1.csv` — raw Ookla sub-tile data

## Step 0: Environment Setup

In [ ]:
# Cell 0.1: Clone repo
import os

REPO = 'satellite-internet-prediction-ML'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

> **Note:** The cell below installs `terratorch` and pins NumPy.  
> After it runs the runtime will **restart automatically**.  
> Re-run from **Step 0.3** after the restart.

In [ ]:
# Cell 0.2: Install dependencies
# TerraTorch MUST be installed first — restart runtime after this cell.

# Pin numpy BEFORE terratorch to prevent 2.x ABI mismatch
!pip install -q "numpy==1.26.4"
!pip install -q terratorch
!pip install -q xgboost rasterio scikit-learn geopandas matplotlib seaborn scipy pyarrow

# Restart runtime so all C extensions load against the pinned numpy
import os
print("Restarting runtime to apply numpy pin — re-run from next cell after restart.")
os.kill(os.getpid(), 9)

In [ ]:
# Cell 0.3: Clone/cd repo (needed again after runtime restart) + sync patches

import os

REPO = 'satellite-internet-prediction-ML'
if not os.getcwd().endswith(REPO):
    if os.path.exists(REPO):
        %cd {REPO}
    else:
        !git clone https://github.com/tatyana21111/{REPO}.git
        %cd {REPO}
!git pull

from google.colab import drive
import shutil
from pathlib import Path

drive.mount('/content/drive', force_remount=False)

PATCH_DIR = 'data/raw/patches_2019'
Path(PATCH_DIR).mkdir(parents=True, exist_ok=True)

local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if local_count < 6900:
    print('Syncing patches from Google Drive ...')
    drive_dir = '/content/drive/MyDrive/patches_2019'
    for f in Path(drive_dir).glob('*.tif'):
        dst = Path(PATCH_DIR) / f.name
        if not dst.exists():
            shutil.copy(f, dst)
    local_count = len(list(Path(PATCH_DIR).glob('*.tif')))
print(f'Patches available: {local_count:,}')


# Verify patches are available
patch_count = len(list(Path(PATCH_DIR).glob('*.tif')))
if patch_count >= 6900:
    print(f'Patches available: {patch_count:,}')
else:
    raise FileNotFoundError(
        f'Only {patch_count} patches found in {PATCH_DIR}. '
        f'Run NB01 first (Steps 1-7) to download patches and save them to Google Drive.'
    )

## Step 1: Imports & Constants

In [ ]:
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
import torch
import xgboost as xgb
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import Normalize
from matplotlib import cm as mpl_cm
import warnings
import logging
import json
from pathlib import Path
from shapely.geometry import box
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
logging.getLogger('rasterio').setLevel(logging.ERROR)

HLS_MEANS  = [0.14245495, 0.13921481, 0.12434631, 0.31420089, 0.20743526, 0.12046503]
HLS_STDS   = [0.04036231, 0.04186983, 0.05267646, 0.08222210, 0.06834774, 0.05294205]
SCALE      = 0.0001
BAND_NAMES = ['Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']

PATCH_DIR      = 'data/raw/patches_2019'
PATCH_SIZE_M   = 6720.0
PATCH_SIZE_DEG = PATCH_SIZE_M / 111_320.0
EARTH_CIRC_M   = 40_075_016.686
ZOOM16_SIDE_EQ = EARTH_CIRC_M / (2 ** 16)

OUTPUT_FEATURES = Path('outputs/features')
OUTPUT_MODELS   = Path('outputs/models')
OUTPUT_FIGURES  = Path('outputs/figures')
OUTPUT_RESULTS  = Path('outputs/results')
for p in [OUTPUT_FEATURES, OUTPUT_FIGURES, OUTPUT_RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {DEVICE}')

## Step 2: Load Data & Train / Val / Test Split
Pre-computed by NB01 (`patches_with_splits.csv`): binary labels, aggregate
targets (`coverage_fraction`, `log_mean_tests`), stratification features,
and geographic split.

In [ ]:
sampled = pd.read_csv('data/processed/patches_with_splits.csv')

# Filter to patches with available TIF files
available = set(f.stem for f in Path(PATCH_DIR).glob('*.tif'))
sampled = sampled[sampled['patch_id'].isin(available)].reset_index(drop=True)

train_df = sampled[sampled['split'] == 'train'].reset_index(drop=True)
val_df   = sampled[sampled['split'] == 'val'].reset_index(drop=True)
test_df  = sampled[sampled['split'] == 'test'].reset_index(drop=True)

print(f'Train: {len(train_df):,}  |  Val: {len(val_df):,}  |  Test (NE): {len(test_df):,}')
print(f'Test binary positive rate   : {test_df["has_coverage"].mean():.2%}')
print(f'Train coverage_fraction mean: {train_df["coverage_fraction"].mean():.4f}')
print(f'Test  coverage_fraction mean: {test_df["coverage_fraction"].mean():.4f}')

## Step 3: Spatial-Resolution Mismatch Visualization

Each HLS chip covers **6.72 × 6.72 km** — roughly **100–130 Ookla zoom-16 sub-tiles**
fit inside a single chip (the exact count depends on latitude via the Mercator
projection).  A binary *"any test observed"* label at the sub-tile level is a
weak supervisory signal at the chip level; the continuous aggregated targets
(`coverage_fraction`, `log_mean_tests`) capture this structure more faithfully.

The figure below shows three example chips from the test set to illustrate
this resolution gap.

In [ ]:
# ── Spatial-resolution mismatch figure ───────────────────────
def total_possible_tiles(lat):
    tile_side = ZOOM16_SIDE_EQ * np.cos(np.radians(lat))
    return int(np.round(PATCH_SIZE_M / tile_side))

# Pick three illustrative test patches: one with high, medium, and low coverage
test_sorted = test_df.sort_values('coverage_fraction').reset_index(drop=True)
low_idx  = test_sorted[test_sorted['coverage_fraction'] > 0].index[0]
mid_idx  = len(test_sorted) // 2
high_idx = len(test_sorted) - 1
examples = test_sorted.iloc[[low_idx, mid_idx, high_idx]]

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
fig.suptitle(
    'Spatial-Resolution Mismatch: 6.72 km HLS Chip vs. Ookla Zoom-16 Sub-tiles',
    fontsize=14, fontweight='bold', y=1.02
)

for ax, (_, row) in zip(axes, examples.iterrows()):
    n_side = total_possible_tiles(row['lat'])
    n_total = n_side ** 2
    n_observed = int(row['n_subtiles'])
    cov_frac = row['coverage_fraction']

    grid = np.zeros((n_side, n_side))
    # Randomly place observed tiles (we don't have exact positions, but this
    # faithfully represents the density)
    rng = np.random.RandomState(42)
    observed_idx = rng.choice(n_total, size=min(n_observed, n_total), replace=False)
    grid.ravel()[observed_idx] = 1

    ax.imshow(grid, cmap='RdYlGn', vmin=0, vmax=1, origin='upper',
              extent=[0, n_side, 0, n_side], interpolation='nearest')

    # Draw chip boundary
    ax.add_patch(plt.Rectangle((0, 0), n_side, n_side, linewidth=2,
                                edgecolor='black', facecolor='none'))

    ax.set_title(
        f'coverage_fraction = {cov_frac:.3f}\n'
        f'{n_observed} / {n_total} sub-tiles observed',
        fontsize=11
    )
    ax.set_xlabel(f'{n_side} tiles × {n_side} tiles  ({PATCH_SIZE_M/1000:.1f} km)', fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#1a9641', edgecolor='black', label='Ookla sub-tile with test data'),
    Patch(facecolor='#d73027', edgecolor='black', label='No test data (unobserved)'),
]
fig.legend(handles=legend_elements, loc='lower center', ncol=2,
           fontsize=11, frameon=True, bbox_to_anchor=(0.5, -0.04))

plt.tight_layout()
plt.savefig(str(OUTPUT_FIGURES / 'spatial_resolution_mismatch.png'),
            dpi=200, bbox_inches='tight')
print(f'Saved: {OUTPUT_FIGURES / "spatial_resolution_mismatch.png"}')
plt.show()

## Step 4: Extract Features for Test Set
We need engineered features (35-d) + Prithvi embeddings (1024-d) for the test set
to generate predictions from the saved NB17 models.

In [ ]:
# 4A: Engineered features
def extract_engineered_features(patch_path):
    with rasterio.open(patch_path) as src:
        img = src.read().astype(np.float32) * SCALE
    feats = {}
    for i, name in enumerate(BAND_NAMES):
        band  = img[i]
        valid = band[band > 0]
        if len(valid) == 0: valid = np.array([0.0])
        feats[f'{name}_mean'] = float(valid.mean())
        feats[f'{name}_std']  = float(valid.std())
        feats[f'{name}_p10']  = float(np.percentile(valid, 10))
        feats[f'{name}_p90']  = float(np.percentile(valid, 90))
    red, nir, swir1, green = img[2], img[3], img[4], img[1]
    ndvi  = np.where((nir+red)    > 0, (nir-red)    / (nir+red    +1e-8), 0)
    ndbi  = np.where((swir1+nir)  > 0, (swir1-nir)  / (swir1+nir  +1e-8), 0)
    mndwi = np.where((green+swir1)> 0, (green-swir1)/ (green+swir1+1e-8), 0)
    feats['NDVI_mean']       = float(ndvi.mean())
    feats['NDVI_std']        = float(ndvi.std())
    feats['NDBI_mean']       = float(ndbi.mean())
    feats['NDBI_std']        = float(ndbi.std())
    feats['MNDWI_mean']      = float(mndwi.mean())
    feats['NIR_spatial_var'] = float(nir.var())
    feats['bright_frac']     = float((red > 0.15).mean())
    return feats


# We need train features too for median imputation
print('Extracting engineered features for train (for median imputation) ...')
rows_train = []
for _, row in tqdm(train_df.iterrows(), total=len(train_df), desc='Train'):
    try:
        rows_train.append(extract_engineered_features(f"{PATCH_DIR}/{row['patch_id']}.tif"))
    except Exception:
        rows_train.append({})
X_train_eng = pd.DataFrame(rows_train)
for feat in ['ntl', 'builtup', 'elevation']:
    X_train_eng[feat] = train_df[feat].values
col_med = X_train_eng.median()

print('Extracting engineered features for test ...')
rows_test = []
for _, row in tqdm(test_df.iterrows(), total=len(test_df), desc='Test'):
    try:
        rows_test.append(extract_engineered_features(f"{PATCH_DIR}/{row['patch_id']}.tif"))
    except Exception:
        rows_test.append({})
X_test_eng = pd.DataFrame(rows_test)
for feat in ['ntl', 'builtup', 'elevation']:
    X_test_eng[feat] = test_df[feat].values
X_test_eng = X_test_eng.fillna(col_med)

ENGINEERED_COLS = list(X_test_eng.columns)
X_test_eng_np = X_test_eng.values.astype(np.float32)
print(f'Engineered test features: {X_test_eng_np.shape}')

In [ ]:
# 4B: Load cached Prithvi embeddings OR extract fresh
emb_path = OUTPUT_FEATURES / 'prithvi_frozen_embeds_test.npz'

if emb_path.exists():
    data = np.load(emb_path)
    X_test_prithvi = data['X'].astype(np.float32)
    print(f'Loaded cached Prithvi test embeddings: {X_test_prithvi.shape}')
else:
    print('Cached embeddings not found — extracting from scratch with TerraTorch ...')

    class PrithviPatchDataset(Dataset):
        def __init__(self, df, patch_dir, n_bands=6):
            self.df        = df.reset_index(drop=True)
            self.patch_dir = patch_dir
            self.n_bands   = n_bands
        def __len__(self):
            return len(self.df)
        def __getitem__(self, idx):
            row  = self.df.iloc[idx]
            path = f"{self.patch_dir}/{row['patch_id']}.tif"
            with rasterio.open(path) as src:
                img = src.read(list(range(1, self.n_bands + 1))).astype(np.float32)
            img *= SCALE
            for b in range(self.n_bands):
                img[b] = (img[b] - HLS_MEANS[b]) / HLS_STDS[b]
            img = np.nan_to_num(img, nan=0.0)
            return torch.from_numpy(img).unsqueeze(1), row['patch_id']

    from terratorch.registry import BACKBONE_REGISTRY

    encoder = BACKBONE_REGISTRY.build("prithvi_eo_v2_300", pretrained=True).to(DEVICE).eval()
    for p in encoder.parameters():
        p.requires_grad = False
    print(f'Prithvi encoder loaded on {DEVICE}')

    @torch.no_grad()
    def extract_prithvi_embeddings(df, batch_size=64):
        ds = PrithviPatchDataset(df, PATCH_DIR)
        loader = DataLoader(ds, batch_size=batch_size, shuffle=False,
                            num_workers=2, pin_memory=True)
        all_embs = []
        for x, _ in tqdm(loader, desc='Extracting Prithvi embeddings'):
            x = x.to(DEVICE, dtype=torch.float32, non_blocking=True)
            feats = encoder(x)
            pooled = feats[-1].mean(dim=1)
            all_embs.append(pooled.cpu().numpy().astype(np.float32))
        return np.concatenate(all_embs, axis=0)

    X_test_prithvi = extract_prithvi_embeddings(test_df)

    # Cache for future runs
    np.savez_compressed(emb_path, X=X_test_prithvi,
                        patch_id=test_df['patch_id'].to_numpy())
    print(f'Embeddings extracted and cached: {X_test_prithvi.shape}')

    del encoder
    torch.cuda.empty_cache()

# Build fusion features
X_test_fusion = np.hstack([X_test_eng_np, X_test_prithvi])
print(f'Prithvi-only test: {X_test_prithvi.shape}')
print(f'Fusion test:       {X_test_fusion.shape}')

## Step 5: Load Saved Models & Generate Predictions

In [ ]:
TARGET = 'log_mean_tests'

# Load NB06 XGBoost models
model_eng = xgb.XGBRegressor()
model_eng.load_model(str(OUTPUT_MODELS / f'fusion_engineered_only_{TARGET}.json'))
print(f'Loaded Engineered-only model: fusion_engineered_only_{TARGET}.json')

model_prithvi = xgb.XGBRegressor()
model_prithvi.load_model(str(OUTPUT_MODELS / f'fusion_prithvi_only_{TARGET}.json'))
print(f'Loaded Prithvi-only model: fusion_prithvi_only_{TARGET}.json')

model_fusion = xgb.XGBRegressor()
model_fusion.load_model(str(OUTPUT_MODELS / f'fusion_engineered_prithvi_{TARGET}.json'))
print(f'Loaded Fusion model: fusion_engineered_prithvi_{TARGET}.json')

# Generate predictions
pred_eng     = model_eng.predict(X_test_eng_np)
pred_prithvi = model_prithvi.predict(X_test_prithvi)
pred_fusion  = model_fusion.predict(X_test_fusion)

print(f'Engineered-only preds    — mean: {pred_eng.mean():.3f}, std: {pred_eng.std():.3f}')
print(f'Prithvi-only predictions — mean: {pred_prithvi.mean():.3f}, std: {pred_prithvi.std():.3f}')
print(f'Fusion predictions       — mean: {pred_fusion.mean():.3f}, std: {pred_fusion.std():.3f}')
print(f'Ground truth             — mean: {test_df[TARGET].mean():.3f}, std: {test_df[TARGET].std():.3f}')

## Step 6: Attach Predictions to Test DataFrame

In [ ]:
test_df['pred_eng']     = pred_eng
test_df['pred_prithvi'] = pred_prithvi
test_df['pred_fusion']  = pred_fusion

print(f'Test set: {len(test_df)} chips in NE India')
print(f'Lon range: [{test_df["lon"].min():.2f}, {test_df["lon"].max():.2f}]')
print(f'Lat range: [{test_df["lat"].min():.2f}, {test_df["lat"].max():.2f}]')
print(f'Ground truth    — mean: {test_df["log_mean_tests"].mean():.3f}')
print(f'Engineered pred — mean: {pred_eng.mean():.3f}')
print(f'Prithvi pred    — mean: {pred_prithvi.mean():.3f}')
print(f'Fusion pred     — mean: {pred_fusion.mean():.3f}')

## Step 7: Install Cartopy & Define Choropleth Helpers
Dark cartographic basemap with large filled patch rectangles — similar to
Jean et al. (2016) poverty-mapping figures. State boundaries via `cartopy.feature.STATES`
provide geographic context without cluttering the overlay.

In [ ]:
!pip install -q cartopy geopandas

import cartopy.crs as ccrs
import cartopy.feature as cfeature
from matplotlib.patches import Rectangle
from matplotlib.collections import PatchCollection
from matplotlib.colors import Normalize, TwoSlopeNorm
import matplotlib.pyplot as plt
import numpy as np

# ── Patch size in degrees ──
VIS_PATCH_DEG = PATCH_SIZE_DEG * 1.8   # enlarge for visibility

# ── Tight extent around actual data ──
PAD = 0.8
EXTENT = [
    test_df['lon'].min() - PAD, test_df['lon'].max() + PAD,
    test_df['lat'].min() - PAD, test_df['lat'].max() + PAD,
]

def make_choropleth(ax, lons, lats, values, cmap, norm, cbar_label, title):
    """Draw filled rectangles on a clean cartographic base."""
    ax.set_extent(EXTENT, crs=ccrs.PlateCarree())

    # ── Clean basemap: gray land, white ocean, thin borders ──
    ax.add_feature(cfeature.OCEAN, facecolor='#1a1a2e', zorder=0)
    ax.add_feature(cfeature.LAND,  facecolor='#2d2d3d', zorder=0)
    ax.add_feature(cfeature.BORDERS, linewidth=0.4, edgecolor='#666666', zorder=1)
    ax.add_feature(cfeature.STATES,  linewidth=0.2, edgecolor='#555555', zorder=1)

    # ── Filled rectangles for each patch ──
    half = VIS_PATCH_DEG / 2
    cm = plt.get_cmap(cmap)
    for lon, lat, val in zip(lons, lats, values):
        color = cm(norm(val))
        rect = Rectangle((lon - half, lat - half), VIS_PATCH_DEG, VIS_PATCH_DEG,
                          facecolor=color, edgecolor='none', alpha=0.85,
                          transform=ccrs.PlateCarree(), zorder=2)
        ax.add_patch(rect)

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cm, norm=norm)
    sm.set_array([])
    cbar = plt.colorbar(sm, ax=ax, fraction=0.035, pad=0.02, shrink=0.85)
    cbar.set_label(cbar_label, fontsize=11, color='white')
    cbar.ax.yaxis.set_tick_params(color='white')
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white', fontsize=9)

    ax.set_title(title, fontsize=13, fontweight='bold', color='white', pad=10)
    ax.set_facecolor('#1a1a2e')


def plot_triptych(lons, lats, y_true, y_pred, target_name, model_label, fignum='1'):
    """Three-panel figure: Observed | Predicted | Residuals."""
    residuals = y_pred - y_true
    vmin, vmax = 0, max(y_true.max(), y_pred.max()) * 0.95

    fig, axes = plt.subplots(1, 3, figsize=(22, 7),
                              subplot_kw={'projection': ccrs.PlateCarree()})
    fig.patch.set_facecolor('#1a1a2e')

    fig.suptitle(
        f'Figure {fignum}: Satellite-Inferred Connectivity — {target_name}\n'
        f'Northeast India Test Set (n = {len(lons)}) — {model_label}',
        fontsize=15, fontweight='bold', color='white', y=0.98
    )

    # (A) Observed
    norm_seq = Normalize(vmin=vmin, vmax=vmax)
    make_choropleth(axes[0], lons, lats, y_true,
                    cmap='RdYlGn', norm=norm_seq,
                    cbar_label=target_name, title='(A) Observed (Ookla Ground Truth)')

    # (B) Predicted
    make_choropleth(axes[1], lons, lats, y_pred,
                    cmap='RdYlGn', norm=norm_seq,
                    cbar_label=f'Predicted {target_name}', title=f'(B) Predicted ({model_label})')

    # (C) Residuals
    res_abs = max(abs(residuals.min()), abs(residuals.max()), 0.01)
    norm_div = TwoSlopeNorm(vmin=-res_abs, vcenter=0, vmax=res_abs)
    make_choropleth(axes[2], lons, lats, residuals,
                    cmap='RdBu_r', norm=norm_div,
                    cbar_label='Residual', title='(C) Residuals (Pred − True)')

    plt.tight_layout(rect=[0, 0, 1, 0.93])

    fname = f'satellite_overlay_{model_label.lower().replace("+","_").replace(" ","_")}.png'
    plt.savefig(str(OUTPUT_FIGURES / fname), dpi=200, bbox_inches='tight',
                facecolor=fig.get_facecolor())
    print(f'Saved: {OUTPUT_FIGURES / fname}')
    plt.show()

## Step 7b: Patch-Level Case Study Grid (Prithvi Paper Fig. 10 Style)

Four-column figure: each row is one example HLS chip from the NE India test
set, selected at quantiles to span no / low / moderate / high connectivity.
Columns show the **HLS RGB composite**, **ground truth**, **engineered-only
prediction**, and **fusion prediction**, each with a colour overlay on the
dimmed satellite image. Residuals are annotated per cell (red = over-prediction,
blue = under-prediction).

In [ ]:
# ═══ Patch-level case study grid (Prithvi paper Fig.10 style) ═══
import matplotlib.patheffects as pe

# ── Select 4 interesting patches by quantile ─────────────────
ne_mask = (test_df['lat'] >= 21) & (test_df['lat'] <= 28.5)
df_ne = test_df[ne_mask].copy()
sorted_ne = df_ne.sort_values(TARGET).reset_index()
n = len(sorted_ne)

cases = {}
cases['No connectivity']       = sorted_ne.iloc[0]['index']
cases['Low connectivity']      = sorted_ne.iloc[int(n * 0.7)]['index']
cases['Moderate connectivity'] = sorted_ne.iloc[int(n * 0.9)]['index']
cases['High connectivity']     = sorted_ne.iloc[-1]['index']

print('Selected case study patches:')
for label, idx in cases.items():
    row = test_df.loc[idx]
    print(f'  {label}: patch={row["patch_id"]}, '
          f'GT={row[TARGET]:.3f}, Eng={row["pred_eng"]:.3f}, '
          f'Fusion={row["pred_fusion"]:.3f}, Prithvi={row["pred_prithvi"]:.3f}')


# ── Read HLS chip as RGB ─────────────────────────────────────
def load_rgb(patch_id, bands=[2, 1, 0]):
    """Load HLS chip and return percentile-stretched RGB."""
    path = f'{PATCH_DIR}/{patch_id}.tif'
    with rasterio.open(path) as src:
        img = src.read().astype(np.float32) * SCALE
    rgb = np.stack([img[b] for b in bands], axis=-1)
    for c in range(3):
        lo, hi = np.percentile(rgb[:, :, c][rgb[:, :, c] > 0], [2, 98])
        rgb[:, :, c] = np.clip((rgb[:, :, c] - lo) / (hi - lo + 1e-8), 0, 1)
    return rgb


# ── Build the figure ─────────────────────────────────────────
n_cases = len(cases)
col_labels = ['HLS input (RGB)', 'Ground truth',
              'Eng-only prediction', 'Fusion prediction']
n_cols = 4

fig, axes = plt.subplots(n_cases, n_cols, figsize=(16, 4 * n_cases))
fig.patch.set_facecolor('white')

vmin, vmax = 0, max(test_df[TARGET].max(), test_df['pred_fusion'].max()) * 0.95
cmap = plt.get_cmap('RdYlGn')
norm = Normalize(vmin=vmin, vmax=vmax)

for row_i, (case_label, idx) in enumerate(cases.items()):
    row = test_df.loc[idx]
    rgb = load_rgb(row['patch_id'])
    h, w = rgb.shape[:2]

    vals = [None, row[TARGET], row['pred_eng'], row['pred_fusion']]
    residuals = [None, None,
                 row['pred_eng'] - row[TARGET],
                 row['pred_fusion'] - row[TARGET]]

    for col_i in range(n_cols):
        ax = axes[row_i, col_i]

        if col_i == 0:
            ax.imshow(rgb, aspect='equal')
        else:
            ax.imshow(rgb, aspect='equal')
            color_val = cmap(norm(vals[col_i]))
            overlay = np.full((*rgb.shape[:2], 4), color_val, dtype=np.float32)
            overlay[:, :, 3] = 0.55
            ax.imshow(overlay, aspect='equal')

            ax.text(w // 2, h * 0.45, f'{vals[col_i]:.3f}',
                    ha='center', va='center', fontsize=22, fontweight='bold',
                    color='white',
                    path_effects=[pe.withStroke(linewidth=3, foreground='black')])

            if residuals[col_i] is not None:
                res = residuals[col_i]
                sign = '+' if res >= 0 else ''
                res_color = '#ff6b6b' if res > 0 else '#4dabf7' if res < 0 else 'white'
                ax.text(w // 2, h * 0.65, f'residual: {sign}{res:.3f}',
                        ha='center', va='center', fontsize=11, color=res_color,
                        path_effects=[pe.withStroke(linewidth=2, foreground='black')])

        ax.set_xticks([])
        ax.set_yticks([])

        if row_i == 0:
            ax.set_title(col_labels[col_i], fontsize=12, fontweight='bold', pad=8)
        if col_i == 0:
            ax.set_ylabel(case_label, fontsize=11, fontweight='bold',
                          rotation=0, labelpad=80, va='center')

# Shared colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label(TARGET, fontsize=12)

fig.suptitle(
    f'Patch-Level Case Study — {TARGET}\n'
    f'Northeast India Test Set: HLS Input vs. Model Predictions',
    fontsize=15, fontweight='bold', y=1.02
)

plt.tight_layout(rect=[0.12, 0, 0.91, 0.96])
plt.savefig(str(OUTPUT_FIGURES / f'case_study_grid_{TARGET}.png'),
            dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved: {OUTPUT_FIGURES / f"case_study_grid_{TARGET}.png"}')
plt.show()

## Step 7c: 3×3 Spatial Mosaic — Neighbourhood Prediction Map

Same four-column layout, but each panel is a **3×3 mosaic of spatially
adjacent patches** stitched together.  This shows how predicted connectivity
varies smoothly across a local area, making colour gradients visible.

In [ ]:
# ═══ 3×3 spatial mosaic figure ════════════════════════════════
from itertools import product

# ── Find the best 3×3 cluster of adjacent patches ────────────
# Patches sit on a regular grid with spacing ~ PATCH_SIZE_DEG.
# We search all test patches for the 3×3 block with the most
# colour variation (std of ground truth), so the figure is informative.

grid_step = PATCH_SIZE_DEG
tol = grid_step * 0.3  # tolerance for snapping to grid

def find_best_3x3_cluster(df, target_col, grid_step, tol):
    """Find the 3×3 block of adjacent patches with highest target std."""
    best_score, best_indices = -1, None

    for anchor_i, anchor in df.iterrows():
        indices = []
        for dr, dc in product(range(3), range(3)):
            target_lon = anchor['lon'] + dc * grid_step
            target_lat = anchor['lat'] - dr * grid_step  # lat decreases downward
            dist = np.sqrt((df['lon'] - target_lon)**2 + (df['lat'] - target_lat)**2)
            nearest = dist.idxmin()
            if dist[nearest] < tol:
                indices.append(nearest)

        if len(set(indices)) == 9:
            vals = df.loc[indices, target_col].values
            score = vals.std()
            if score > best_score:
                best_score = score
                best_indices = indices

    return best_indices

print('Searching for best 3×3 cluster (this may take a moment) ...')
cluster_idx = find_best_3x3_cluster(test_df, TARGET, grid_step, tol)

if cluster_idx is None:
    print('No perfect 3×3 block found — falling back to 9 patches with most variation.')
    cluster_idx = test_df.nlargest(3, TARGET).index.tolist() + \
                  test_df.iloc[(test_df[TARGET] - test_df[TARGET].median()).abs().argsort()[:3]].index.tolist() + \
                  test_df.nsmallest(3, TARGET).index.tolist()
    cluster_idx = cluster_idx[:9]

cluster = test_df.loc[cluster_idx].copy()
cluster = cluster.sort_values(['lat', 'lon'], ascending=[False, True]).reset_index(drop=True)

print(f'Cluster of {len(cluster)} patches:')
print(f'  Lon: [{cluster["lon"].min():.3f}, {cluster["lon"].max():.3f}]')
print(f'  Lat: [{cluster["lat"].min():.3f}, {cluster["lat"].max():.3f}]')
print(f'  {TARGET} range: [{cluster[TARGET].min():.3f}, {cluster[TARGET].max():.3f}]')


# ── Stitch a 3×3 mosaic ──────────────────────────────────────
def stitch_mosaic(patches_3x3, load_fn, overlay_vals=None, cmap=None, norm=None, alpha=0.55):
    """Stitch 9 RGB arrays into a 3×3 grid; optionally add colour overlay."""
    rows_out = []
    for r in range(3):
        row_imgs = []
        for c in range(3):
            i = r * 3 + c
            rgb = load_fn(patches_3x3.iloc[i]['patch_id'])

            if overlay_vals is not None:
                val = overlay_vals[i]
                color_val = cmap(norm(val))
                ov = np.full((*rgb.shape[:2], 4), color_val, dtype=np.float32)
                ov[:, :, 3] = alpha
                # Composite: dim the satellite + blend overlay
                rgb_dimmed = rgb * 0.35 + 0.1
                # Manual alpha composite
                rgb_out = rgb_dimmed * (1 - alpha) + np.array(color_val[:3]) * alpha
                rgb_out = np.clip(rgb_out, 0, 1)
            else:
                rgb_out = rgb

            row_imgs.append(rgb_out)
        rows_out.append(np.concatenate(row_imgs, axis=1))
    return np.concatenate(rows_out, axis=0)


# ── Build the 4-column figure ────────────────────────────────
vmin, vmax = 0, max(test_df[TARGET].max(), test_df['pred_fusion'].max()) * 0.95
cmap = plt.get_cmap('RdYlGn')
norm = Normalize(vmin=vmin, vmax=vmax)

gt_vals  = cluster[TARGET].values
eng_vals = cluster['pred_eng'].values
fus_vals = cluster['pred_fusion'].values

mosaic_rgb = stitch_mosaic(cluster, load_rgb)
mosaic_gt  = stitch_mosaic(cluster, load_rgb, overlay_vals=gt_vals,  cmap=cmap, norm=norm)
mosaic_eng = stitch_mosaic(cluster, load_rgb, overlay_vals=eng_vals, cmap=cmap, norm=norm)
mosaic_fus = stitch_mosaic(cluster, load_rgb, overlay_vals=fus_vals, cmap=cmap, norm=norm)

fig, axes = plt.subplots(1, 4, figsize=(22, 6))
fig.patch.set_facecolor('white')

col_data = [
    ('HLS input (RGB)',        mosaic_rgb),
    ('Ground truth',           mosaic_gt),
    ('Eng-only prediction',    mosaic_eng),
    ('Fusion prediction',      mosaic_fus),
]

for ax, (title, mosaic) in zip(axes, col_data):
    ax.imshow(mosaic, aspect='equal')
    ax.set_title(title, fontsize=13, fontweight='bold', pad=8)
    ax.set_xticks([])
    ax.set_yticks([])

    # Draw grid lines between patches
    h_patch = mosaic.shape[0] // 3
    w_patch = mosaic.shape[1] // 3
    for k in range(1, 3):
        ax.axhline(k * h_patch, color='white', linewidth=0.8, alpha=0.6)
        ax.axvline(k * w_patch, color='white', linewidth=0.8, alpha=0.6)

# Shared colorbar
sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
sm.set_array([])
cbar = fig.colorbar(sm, ax=axes.tolist(), fraction=0.02, pad=0.02, shrink=0.85)
cbar.set_label(TARGET, fontsize=12)

fig.suptitle(
    f'3×3 Spatial Mosaic — {TARGET}\n'
    f'Lon [{cluster["lon"].min():.2f}, {cluster["lon"].max():.2f}], '
    f'Lat [{cluster["lat"].min():.2f}, {cluster["lat"].max():.2f}]',
    fontsize=14, fontweight='bold', y=1.04
)

plt.tight_layout()
plt.savefig(str(OUTPUT_FIGURES / f'mosaic_3x3_{TARGET}.png'),
            dpi=200, bbox_inches='tight', facecolor='white')
print(f'Saved: {OUTPUT_FIGURES / f"mosaic_3x3_{TARGET}.png"}')
plt.show()

## Step 8: Main Overlay Figures — Connectivity Heatmaps
Three-panel figures (one per model), each showing the **entire NE India test set**:
- **(A) Observed** (`log_mean_tests` ground truth) — `RdYlGn`, green = high connectivity
- **(B) Predicted** (model output) — same colormap & scale
- **(C) Residuals** (predicted − true) — `RdBu_r` diverging, blue = under-prediction

In [ ]:
# ── Prithvi-only model ──
plot_triptych(
    test_df['lon'].values, test_df['lat'].values,
    test_df['log_mean_tests'].values, test_df['pred_prithvi'].values,
    target_name='log_mean_tests',
    model_label='Prithvi Only',
    fignum='1'
)

# ── Engineered + Prithvi Fusion model ──
plot_triptych(
    test_df['lon'].values, test_df['lat'].values,
    test_df['log_mean_tests'].values, test_df['pred_fusion'].values,
    target_name='log_mean_tests',
    model_label='Eng.+Prithvi Fusion',
    fignum='2'
)

## Step 9: Binary Overlay Version
2-panel figure: **(A) Observed** binary coverage vs **(B) Predicted** binary coverage
(fusion model at val-optimal threshold).

In [ ]:
# Load val-optimal thresholds from NB06 metrics
def load_threshold(model_key, target):
    path = OUTPUT_RESULTS / f'fusion_{model_key}_{target}_metrics.csv'
    if path.exists():
        return pd.read_csv(path)['opt_threshold'].iloc[0]
    return None

thr_fusion = load_threshold('engineered_prithvi', 'log_mean_tests')
print(f'Fusion threshold (log_mean_tests): {thr_fusion}')

if thr_fusion is not None:
    obs_bin  = test_df['has_coverage'].values
    pred_bin = (test_df['pred_fusion'].values >= thr_fusion).astype(int)

    fig, axes = plt.subplots(1, 2, figsize=(16, 7),
                              subplot_kw={'projection': ccrs.PlateCarree()})
    fig.patch.set_facecolor('#1a1a2e')
    fig.suptitle(
        'Binary Connectivity Overlay — NE India Test Set\n'
        f'Fusion model @ threshold = {thr_fusion:.3f}',
        fontsize=14, fontweight='bold', color='white', y=0.98
    )

    norm_bin = Normalize(vmin=0, vmax=1)
    make_choropleth(axes[0], test_df['lon'].values, test_df['lat'].values, obs_bin,
                    cmap='RdYlGn', norm=norm_bin,
                    cbar_label='has_coverage', title='(A) Observed (Ground Truth)')
    make_choropleth(axes[1], test_df['lon'].values, test_df['lat'].values, pred_bin,
                    cmap='RdYlGn', norm=norm_bin,
                    cbar_label='Predicted', title='(B) Predicted (Fusion)')

    plt.tight_layout(rect=[0, 0, 1, 0.93])
    plt.savefig(str(OUTPUT_FIGURES / 'satellite_overlay_binary.png'), dpi=200,
                bbox_inches='tight', facecolor=fig.get_facecolor())
    print(f'Saved: {OUTPUT_FIGURES / "satellite_overlay_binary.png"}')
    plt.show()
else:
    print('Threshold not found — run NB06 first.')

## Step 12: Summary Statistics — Full NE India Test Set

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
from scipy.stats import spearmanr

gt = test_df[TARGET].values

rows = []
for model_name, preds in [('Prithvi-only', pred_prithvi), ('Fusion', pred_fusion)]:
    residuals = preds - gt
    rows.append({
        'Model': model_name,
        'N chips': len(gt),
        'MAE': mean_absolute_error(gt, preds),
        'RMSE': np.sqrt(mean_squared_error(gt, preds)),
        'Spearman rho': spearmanr(gt, preds).statistic,
        'Mean residual': residuals.mean(),
        'Residual std': residuals.std(),
    })

summary = pd.DataFrame(rows)
print('=== NE India Test Set Summary ===')
print(summary.to_string(index=False))

summary.to_csv(OUTPUT_RESULTS / 'nb20_test_summary.csv', index=False)
print(f'\nSaved: {OUTPUT_RESULTS / "nb20_test_summary.csv"}')

## Step 13: Push to GitHub

In [ ]:
import subprocess, os

token = "YOUR_TOKEN_HERE"
repo  = "tatyana21111/satellite-internet-prediction-ML"

subprocess.run(['git', 'config', '--global', 'user.email', 'tatyana211zy@outlook.com'], check=True)
subprocess.run(['git', 'config', '--global', 'user.name',  'tatyana21111'], check=True)
subprocess.run(['git', 'remote', 'set-url', 'origin',
                f'https://{token}@github.com/{repo}.git'], check=True)

# Add this notebook
subprocess.run(['git', 'add', '-f', 'notebooks/07_qualitative_analysis.ipynb'], check=True)

# Add all figures and results produced by this notebook
for pat in ['*.png', '*.pdf']:
    for p in OUTPUT_FIGURES.glob(pat):
        subprocess.run(['git', 'add', '-f', str(p)], check=True)
        print(f'  Staged: {p}')

for pat in ['*.csv']:
    for p in OUTPUT_RESULTS.glob(pat):
        subprocess.run(['git', 'add', '-f', str(p)], check=True)
        print(f'  Staged: {p}')

diff = subprocess.run(['git', 'diff', '--cached', '--quiet'], capture_output=True)
if diff.returncode != 0:
    subprocess.run(['git', 'commit', '-m',
                    'NB07: Qualitative analysis — choropleth maps, case study grid, resolution mismatch'], check=True)
else:
    print('Nothing staged.')

subprocess.run(['git', 'pull', '--rebase', 'origin', 'main'], check=True)
subprocess.run(['git', 'push'], check=True)
print('Push successful.')